# PaddleOCR-VL-1.6 configurable L4 benchmark

This is one inference-only notebook for the three study variants. Make a
copy for each Kaggle account and change only `SELECTED_VARIANT`.

Always run `PREFLIGHT` first. It executes the complete c1/c24 workflow on
nine fixed pages, including fresh server launches, GPU telemetry, output
validation, and archive creation. After it prints `PREFLIGHT: PASS`, change
`RUN_MODE` to `FULL` and run the notebook from the beginning.

Locked topology: GPU 0 runs vLLM, GPU 1 runs PP-DocLayoutV3, and GPUs 2–3
are intentionally unused. Each concurrency gets a fresh vLLM process.


In [ ]:
# ================================================================
# CELL 1 — USER SETTINGS
# Change only these two values for normal use.
# ================================================================

SELECTED_VARIANT = 'BF16'       # BF16 | W8A16 | W4A16
RUN_MODE = 'PREFLIGHT'          # PREFLIGHT | FULL

# Optional: set this only if automatic checkpoint discovery is ambiguous.
# Example: '/kaggle/input/my-checkpoints/PaddleOCR-VL-1.6-GPTQ-W8A16-G128'
CHECKPOINT_OVERRIDE = None

CONCURRENCIES = [1, 24]
TELEMETRY_INTERVAL_MS = 200
CLEAN_TEMP_ENVIRONMENTS_AFTER_RUN = True

# Nine pages distributed across the complete 1,651-page manifest.
PREFLIGHT_INDICES = [0, 103, 309, 584, 825, 1066, 1341, 1547, 1650]
WARMUP_INDICES = [34, 859, 1616]

assert SELECTED_VARIANT in {'BF16', 'W8A16', 'W4A16'}
assert RUN_MODE in {'PREFLIGHT', 'FULL'}
assert CONCURRENCIES == [1, 24]
assert len(PREFLIGHT_INDICES) == len(set(PREFLIGHT_INDICES)) == 9
assert TELEMETRY_INTERVAL_MS >= 100

print('Selected variant:', SELECTED_VARIANT)
print('Run mode:', RUN_MODE)


In [ ]:
# ================================================================
# CELL 2 — IMPORTS, OUTPUT LOCATION, BUNDLE AND HARDWARE
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import bisect
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import signal
import socket
import statistics
import subprocess
import sys
import time
import urllib.request
import zipfile


EXPECTED_PAGES = 1651
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
VARIANT_SLUG = SELECTED_VARIANT.lower()
MODE_SLUG = RUN_MODE.lower()

OUTPUT_ROOT = Path('/kaggle/working') / (
    f'l4_wna16_{VARIANT_SLUG}_{MODE_SLUG}_{RUN_TAG}'
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

SERVER_SITE = Path('/kaggle/working/l4_wna16_server_site')
CLIENT_SITE = Path('/kaggle/working/l4_wna16_client_site')
CLIENT_WORKER = Path('/kaggle/working/l4_wna16_full_client_worker.py')


def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


manifest_candidates = []
for path in Path('/kaggle/input').rglob('bundle_manifest.json'):
    try:
        payload = load_json(path)
    except Exception:
        continue
    if payload.get('bundle') == 'paddleocr_vl_1_6_l4_wna16_bundle_v1':
        manifest_candidates.append(path)

if len(manifest_candidates) != 1:
    raise RuntimeError(
        'Expected exactly one attached dependency bundle; '
        f'found {manifest_candidates}'
    )

MANIFEST_PATH = manifest_candidates[0]
BUNDLE = MANIFEST_PATH.parent
MANIFEST = load_json(MANIFEST_PATH)
STUDY = load_json(BUNDLE / MANIFEST['paths']['study_design'])
RUNTIME_SOURCE = load_json(BUNDLE / MANIFEST['paths']['runtime_contract'])

gpu_query = subprocess.check_output(
    [
        'nvidia-smi',
        '--query-gpu=index,name,compute_cap,memory.total,uuid,driver_version',
        '--format=csv,noheader,nounits',
    ],
    text=True,
)
gpu_rows = [row for row in csv.reader(gpu_query.splitlines()) if row]

if len(gpu_rows) != 4:
    raise RuntimeError(f'Expected four GPUs, got {len(gpu_rows)}')
if not all('L4' in row[1] for row in gpu_rows):
    raise RuntimeError(f'Expected four NVIDIA L4 GPUs: {gpu_rows}')
if not all(row[2].strip() == '8.9' for row in gpu_rows):
    raise RuntimeError(f'Expected compute capability 8.9: {gpu_rows}')

expected_assignment = {
    'vllm_server_physical_gpu': 0,
    'paddle_layout_physical_gpu': 1,
    'unused_physical_gpus': [2, 3],
}
if RUNTIME_SOURCE['gpu_assignment'] != expected_assignment:
    raise RuntimeError('GPU assignment contract drift')

print('Output root:', OUTPUT_ROOT)
print('Bundle:', BUNDLE)
print(gpu_query)
print('BUNDLE + HARDWARE: PASS')


In [ ]:
# ================================================================
# CELL 3 — DATASET, CHECKPOINT AND RUN-CONTRACT VALIDATION
# ================================================================

BENCHMARK_INPUTS = BUNDLE / MANIFEST['paths']['benchmark_inputs']
LAYOUT = BUNDLE / MANIFEST['paths']['layout']
BF16 = BUNDLE / MANIFEST['paths']['bf16']
OMNIDOCBENCH = BUNDLE / MANIFEST['paths']['omnidocbench']
GROUND_TRUTH = OMNIDOCBENCH / 'OmniDocBench.json'

for required in [BENCHMARK_INPUTS, GROUND_TRUTH, BF16 / 'config.json']:
    if not required.is_file():
        raise RuntimeError(f'Required file missing: {required}')
for required in [
    LAYOUT / 'inference.json',
    LAYOUT / 'inference.pdiparams',
    LAYOUT / 'inference.yml',
]:
    if not required.is_file():
        raise RuntimeError(f'Layout file missing: {required}')

benchmark_inputs = load_json(BENCHMARK_INPUTS)
ground_truth = load_json(GROUND_TRUTH)
if len(benchmark_inputs) != EXPECTED_PAGES or len(ground_truth) != EXPECTED_PAGES:
    raise RuntimeError('OmniDocBench page-count drift')

page_ids = []
expected_markdown_names = []
for position, entry in enumerate(benchmark_inputs):
    index = int(entry.get('annotation_index', position))
    if not 0 <= index < EXPECTED_PAGES:
        raise RuntimeError(f'Invalid annotation index at row {position}')
    page_id = entry['page_id']
    image_path = BUNDLE / entry['image']
    if not image_path.is_file():
        raise RuntimeError(f'Benchmark image missing: {image_path}')
    image_reference = ground_truth[index].get('page_info', {}).get('image_path')
    if not isinstance(image_reference, str) or not image_reference:
        raise RuntimeError(f'Ground-truth image path missing at {index}')
    ground_truth_name = image_reference.replace('\\', '/').rsplit('/', 1)[-1]
    if Path(ground_truth_name).stem != page_id:
        raise RuntimeError(
            f'Prediction-name mismatch at {index}: '
            f'{page_id!r} vs {ground_truth_name!r}'
        )
    page_ids.append(page_id)
    expected_markdown_names.append(page_id + '.md')

if len(set(page_ids)) != EXPECTED_PAGES:
    raise RuntimeError('OmniDocBench page IDs are not unique')
if len(set(expected_markdown_names)) != EXPECTED_PAGES:
    raise RuntimeError('Evaluator Markdown names are not unique')

if RUN_MODE == 'PREFLIGHT':
    MEASURED_INDICES = list(PREFLIGHT_INDICES)
else:
    MEASURED_INDICES = list(range(EXPECTED_PAGES))

if len(MEASURED_INDICES) != len(set(MEASURED_INDICES)):
    raise RuntimeError('Measured indices are not unique')
if not all(0 <= index < EXPECTED_PAGES for index in MEASURED_INDICES):
    raise RuntimeError('Measured index outside the dataset')

variant_specs = {
    'BF16': {
        'id': 'BF16_W16A16',
        'directory_name': BF16.name,
        'quantized': False,
        'weight_bits': 16,
        'weights': 'BF16',
        'activations': 'BF16',
    },
    'W8A16': {
        'id': 'GPTQ_W8A16_G128',
        'directory_name': MANIFEST['quantized_output_names']['W8A16'],
        'quantized': True,
        'weight_bits': 8,
        'weights': 'INT8',
        'activations': 'BF16',
    },
    'W4A16': {
        'id': 'GPTQ_W4A16_G128',
        'directory_name': MANIFEST['quantized_output_names']['W4A16'],
        'quantized': True,
        'weight_bits': 4,
        'weights': 'INT4',
        'activations': 'BF16',
    },
}
VARIANT = dict(variant_specs[SELECTED_VARIANT])

if CHECKPOINT_OVERRIDE:
    CHECKPOINT = Path(CHECKPOINT_OVERRIDE)
elif SELECTED_VARIANT == 'BF16':
    CHECKPOINT = BF16
else:
    checkpoint_candidates = sorted({
        path.parent
        for path in Path('/kaggle/input').rglob('config.json')
        if path.parent.name == VARIANT['directory_name']
        and list(path.parent.glob('*.safetensors'))
    })
    if len(checkpoint_candidates) != 1:
        raise RuntimeError(
            f'Expected one attached {SELECTED_VARIANT} checkpoint named '
            f'{VARIANT["directory_name"]!r}; found {checkpoint_candidates}. '
            'Attach the checkpoint output or set CHECKPOINT_OVERRIDE.'
        )
    CHECKPOINT = checkpoint_candidates[0]

if not CHECKPOINT.is_dir() or not (CHECKPOINT / 'config.json').is_file():
    raise RuntimeError(f'Checkpoint is incomplete: {CHECKPOINT}')
safetensors = sorted(CHECKPOINT.glob('*.safetensors'))
if not safetensors or any(path.stat().st_size == 0 for path in safetensors):
    raise RuntimeError(f'Checkpoint has no valid safetensors: {CHECKPOINT}')


def collect_nested_values(value, requested_key):
    found = []
    if isinstance(value, dict):
        for key, child in value.items():
            if key == requested_key:
                found.append(child)
            found.extend(collect_nested_values(child, requested_key))
    elif isinstance(value, list):
        for child in value:
            found.extend(collect_nested_values(child, requested_key))
    return found


checkpoint_config = load_json(CHECKPOINT / 'config.json')
scope_audit = None
if VARIANT['quantized']:
    audit_path = CHECKPOINT / 'scope_audit.json'
    if not audit_path.is_file():
        raise RuntimeError(f'Quantized checkpoint audit missing: {audit_path}')
    scope_audit = load_json(audit_path)
    required_audit = {
        'algorithm': 'GPTQ',
        'group_size': 128,
        'activation_dtype': 'BF16',
        'calibration_count': 512,
        'target_count': 234,
    }
    for key, expected in required_audit.items():
        if scope_audit.get(key) != expected:
            raise RuntimeError(
                f'Checkpoint audit mismatch for {key}: '
                f'{scope_audit.get(key)!r} != {expected!r}'
            )
    if scope_audit.get('vision_layer_indices') != list(range(27)):
        raise RuntimeError('Vision quantization scope drift')
    if scope_audit.get('decoder_layer_indices') != list(range(18)):
        raise RuntimeError('Decoder quantization scope drift')
    quantization_config = checkpoint_config.get('quantization_config')
    if not isinstance(quantization_config, dict):
        raise RuntimeError('Missing quantization_config')
    bits = {
        int(value)
        for value in collect_nested_values(quantization_config, 'num_bits')
        if value is not None
    }
    groups = {
        int(value)
        for value in collect_nested_values(quantization_config, 'group_size')
        if value is not None
    }
    if VARIANT['weight_bits'] not in bits or 128 not in groups:
        raise RuntimeError(
            f'Quantization config mismatch: bits={bits}, groups={groups}'
        )

checkpoint_manifest = []
for path in [CHECKPOINT / 'config.json', *safetensors]:
    checkpoint_manifest.append({
        'filename': path.name,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })
if scope_audit is not None:
    path = CHECKPOINT / 'scope_audit.json'
    checkpoint_manifest.append({
        'filename': path.name,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })

selected_contract = {
    'variant_key': SELECTED_VARIANT,
    'variant_id': VARIANT['id'],
    'weights': VARIANT['weights'],
    'activations': VARIANT['activations'],
    'checkpoint': str(CHECKPOINT),
    'checkpoint_files': checkpoint_manifest,
    'checkpoint_total_bytes': sum(item['bytes'] for item in checkpoint_manifest),
    'quantization': (
        {
            'algorithm': 'GPTQ',
            'group_size': 128,
            'calibration_count': 512,
            'logical_target_count': 234,
            'expected_fused_runtime_linear_modules': 126,
            'requested_backend': 'marlin',
        }
        if VARIANT['quantized']
        else {'algorithm': None, 'requested_backend': None}
    ),
    'run_mode': RUN_MODE,
    'concurrencies': CONCURRENCIES,
    'measured_indices': MEASURED_INDICES,
    'warmup_indices': WARMUP_INDICES,
    'expected_prediction_filenames': [
        expected_markdown_names[index] for index in MEASURED_INDICES
    ],
}
(OUTPUT_ROOT / 'selected_contract.json').write_text(
    json.dumps(selected_contract, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print('Variant ID:', VARIANT['id'])
print('Checkpoint:', CHECKPOINT)
print('Measured pages per run:', len(MEASURED_INDICES))
print('DATASET + CHECKPOINT CONTRACT: PASS')


In [ ]:
# ================================================================
# CELL 4 — VERIFY WHEELHOUSES AND INSTALL ISOLATED ENVIRONMENTS
# ================================================================

SERVER_WHEELS = BUNDLE / MANIFEST['paths']['server_wheels']
CLIENT_WHEELS = BUNDLE / MANIFEST['paths']['client_wheels']


def verify_wheelhouse(group, directory):
    if group not in MANIFEST['wheelhouse']:
        raise RuntimeError(
            f'Wheelhouse group {group!r} missing; '
            f'available={sorted(MANIFEST["wheelhouse"])}'
        )
    expected = {
        item['filename']: item
        for item in MANIFEST['wheelhouse'][group]
    }
    actual = {path.name: path for path in Path(directory).glob('*.whl')}
    if set(actual) != set(expected):
        raise RuntimeError(f'{group} wheel filename set drift')
    for name, path in actual.items():
        if path.stat().st_size != expected[name]['bytes']:
            raise RuntimeError(f'Wheel size mismatch: {path}')
        if sha256_file(path) != expected[name]['sha256']:
            raise RuntimeError(f'Wheel hash mismatch: {path}')
        with zipfile.ZipFile(path) as archive:
            if archive.testzip() is not None:
                raise RuntimeError(f'Corrupt wheel: {path}')


verify_wheelhouse('vllm_server', SERVER_WHEELS)
verify_wheelhouse('paddle_client', CLIENT_WHEELS)

for site in [SERVER_SITE, CLIENT_SITE]:
    shutil.rmtree(site, ignore_errors=True)


def run_checked(command, **kwargs):
    print('+', ' '.join(map(str, command)), flush=True)
    return subprocess.run(command, check=True, **kwargs)


run_checked([
    sys.executable, '-m', 'pip', 'install',
    '--no-index', '--pre', '--no-cache-dir',
    '--find-links', str(SERVER_WHEELS),
    '--target', str(SERVER_SITE),
    '--upgrade', '--ignore-installed', '--no-compile',
    'vllm==0.29.0',
])

run_checked([
    sys.executable, '-m', 'pip', 'install',
    '--no-index', '--pre', '--no-cache-dir',
    '--find-links', str(CLIENT_WHEELS),
    '--target', str(CLIENT_SITE),
    '--upgrade', '--ignore-installed', '--no-compile',
    'paddlepaddle-gpu==3.2.1',
    'paddleocr[doc-parser]==3.7.0',
    'paddlex[ocr-core]==3.7.2',
    'numpy==2.2.6',
])

# PaddleX imports ModelScope for optional downloads. Local models are used
# here, and ModelScope can pull in the incompatible global PyTorch build.
official_models_path = (
    CLIENT_SITE / 'paddlex' / 'inference' / 'utils' / 'official_models.py'
)
if not official_models_path.is_file():
    raise RuntimeError(f'PaddleX file missing: {official_models_path}')
official_source = official_models_path.read_text(encoding='utf-8')
patch_marker = 'L4_WNA16_MODELSCOPE_OPTIONAL_IMPORT'
if patch_marker not in official_source:
    patched_lines = []
    replacement_count = 0
    for line in official_source.splitlines():
        if line.strip() == 'import modelscope':
            indentation = line[:len(line) - len(line.lstrip())]
            patched_lines.extend([
                f'{indentation}# {patch_marker}',
                f'{indentation}try:',
                f'{indentation}    import modelscope',
                f'{indentation}except Exception:',
                f'{indentation}    modelscope = None',
            ])
            replacement_count += 1
        else:
            patched_lines.append(line)
    if replacement_count < 1:
        raise RuntimeError('No ModelScope import found to patch')
    patched_source = '\n'.join(patched_lines) + '\n'
    compile(patched_source, str(official_models_path), 'exec')
    official_models_path.write_text(patched_source, encoding='utf-8')
    print('ModelScope imports patched:', replacement_count)

offline_common = {
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_DATASETS_OFFLINE': '1',
    'PIP_NO_INDEX': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTHONHASHSEED': '2026',
    'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK': 'True',
    'NO_PROXY': '127.0.0.1,localhost',
    'no_proxy': '127.0.0.1,localhost',
}

server_verify_env = os.environ.copy()
server_verify_env.update(offline_common)
server_verify_env.update({
    'PYTHONPATH': str(SERVER_SITE),
    'CUDA_VISIBLE_DEVICES': '0',
})
client_verify_env = os.environ.copy()
client_verify_env.update(offline_common)
client_verify_env.update({
    'PYTHONPATH': str(CLIENT_SITE),
    'CUDA_VISIBLE_DEVICES': '1',
})

run_checked([
    sys.executable, '-c',
    "import importlib.metadata as m;import torch,vllm;"
    "assert torch.cuda.is_available();"
    "assert torch.cuda.get_device_name(0).endswith('L4');"
    "assert torch.cuda.get_device_capability(0)==(8,9);"
    "assert m.version('vllm')=='0.29.0';"
    "print(torch.__version__,vllm.__version__,m.version('compressed-tensors'))",
], env=server_verify_env)

run_checked([
    sys.executable, '-c',
    "import paddle,paddleocr,paddlex;"
    "assert paddle.device.cuda.device_count()==1;"
    "paddle.set_device('gpu:0');"
    "print(paddle.__version__,paddleocr.__version__,paddlex.__version__,paddle.device.get_device())",
], env=client_verify_env)

version_probe = (
    "import importlib.metadata as m,json,sys;"
    "names=json.loads(sys.argv[1]);"
    "print(json.dumps({name:m.version(name) for name in names}))"
)
server_distributions = [
    'vllm', 'torch', 'transformers',
    'compressed-tensors', 'safetensors',
]
client_distributions = [
    'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy',
]
SERVER_SOFTWARE = json.loads(subprocess.check_output(
    [
        sys.executable, '-c', version_probe,
        json.dumps(server_distributions),
    ],
    env=server_verify_env,
    text=True,
))
CLIENT_SOFTWARE = json.loads(subprocess.check_output(
    [
        sys.executable, '-c', version_probe,
        json.dumps(client_distributions),
    ],
    env=client_verify_env,
    text=True,
))
SOFTWARE_VERSIONS = {
    'server_environment': SERVER_SOFTWARE,
    'client_environment': CLIENT_SOFTWARE,
}
(OUTPUT_ROOT / 'software_versions.json').write_text(
    json.dumps(SOFTWARE_VERSIONS, indent=2) + '\n',
    encoding='utf-8',
)

print('OFFLINE ENVIRONMENTS: PASS')


In [ ]:
# ================================================================
# CELL 5A — CLIENT WORKER: IMPORTS, CONTRACT AND PROXY
# ================================================================

WORKER_SOURCE_PARTS = []
WORKER_SOURCE_PARTS.append(r'''
import csv
import hashlib
import json
import os
import random
import statistics
import threading
import time
import traceback
import urllib.error
import urllib.request
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import numpy as np
import paddle
from paddleocr import PaddleOCRVL

BUNDLE = Path(os.environ['BUNDLE'])
RESULT_DIR = Path(os.environ['RESULT_DIR'])
LAYOUT_DIR = Path(os.environ['LAYOUT_DIR'])
SERVER_URL = os.environ['SERVER_URL'].rstrip('/')
PROXY_PORT = int(os.environ['PROXY_PORT'])
SERVED_NAME = os.environ['SERVED_NAME']
VARIANT = os.environ['VARIANT']
CONCURRENCY = int(os.environ['REQUEST_CONCURRENCY'])
RUNTIME = json.loads(Path(os.environ['RUNTIME_CONTRACT']).read_text(encoding='utf-8'))
INPUTS = json.loads(Path(os.environ['BENCHMARK_INPUTS']).read_text(encoding='utf-8'))
WARMUP_INDICES = json.loads(os.environ['WARMUP_INDICES_JSON'])
MEASURED_INDICES = json.loads(os.environ['MEASURED_INDICES_JSON'])
EXPECTED_DATASET_PAGES = int(os.environ['EXPECTED_DATASET_PAGES'])
MEASUREMENT_GATE = Path(os.environ['MEASUREMENT_GATE'])
PROGRESS_EVERY = int(os.environ.get('PROGRESS_EVERY', '25'))

if len(INPUTS) != EXPECTED_DATASET_PAGES:
    raise RuntimeError(f'Expected {EXPECTED_DATASET_PAGES} inputs, got {len(INPUTS)}')
if len(MEASURED_INDICES) != len(set(MEASURED_INDICES)):
    raise RuntimeError('Measured indices are not unique')
if not all(0 <= index < len(INPUTS) for index in MEASURED_INDICES):
    raise RuntimeError('Measured index is outside the manifest')
if not all(0 <= index < len(INPUTS) for index in WARMUP_INDICES):
    raise RuntimeError('Warmup index is outside the manifest')

RESULT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATOR_DIR = RESULT_DIR / 'evaluator_input'
RAW_PAGES_DIR = RESULT_DIR / 'raw_pages'
EVALUATOR_DIR.mkdir(parents=True, exist_ok=True)
RAW_PAGES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RESULT_DIR / 'page_metrics.jsonl'
REQUEST_LOG_PATH = RESULT_DIR / 'vlm_request_log.jsonl'

random.seed(2026)
np.random.seed(2026)
paddle.disable_static()
paddle.set_device('gpu:0')
paddle.seed(2026)

PREDICT_KWARGS = {
    'min_pixels': int(RUNTIME['generation']['min_pixels']),
    'max_pixels': int(RUNTIME['generation']['max_pixels']),
    'max_new_tokens': int(RUNTIME['generation']['max_new_tokens']),
    'repetition_penalty': 1.0,
    'temperature': 0.0,
    'top_p': 1.0,
}

def atomic_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(text, encoding='utf-8')
    temporary.replace(path)

def atomic_json(path, payload):
    atomic_text(
        path,
        json.dumps(payload, indent=2, ensure_ascii=False, default=str) + '\n',
    )

def append_jsonl(path, rows):
    if isinstance(rows, dict):
        rows = [rows]
    with Path(path).open('a', encoding='utf-8') as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')
        stream.flush()

proxy_state = {
    'page_id': None,
    'manifest_index': None,
    'phase': None,
    'attempt_id': None,
}
proxy_lock = threading.Lock()
request_records = []
request_counter = 0
upstream_base = SERVER_URL.removesuffix('/v1')

class ProxyHandler(BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'

    def log_message(self, fmt, *args):
        return

    def do_GET(self):
        self.forward()

    def do_POST(self):
        self.forward()

    def forward(self):
        global request_counter
        body = self.rfile.read(int(self.headers.get('Content-Length', '0') or 0))
        with proxy_lock:
            request_counter += 1
            request_id = request_counter
            context = dict(proxy_state)
        started = time.perf_counter()
        status = 599
        response_body = b''
        content_type = 'application/json'
        error = None
        try:
            request = urllib.request.Request(
                upstream_base + self.path,
                data=body if self.command != 'GET' else None,
                headers={
                    'Content-Type': self.headers.get(
                        'Content-Type', 'application/json'
                    )
                },
                method=self.command,
            )
            with urllib.request.urlopen(request, timeout=1800) as response:
                status = response.status
                response_body = response.read()
                content_type = response.headers.get(
                    'Content-Type', 'application/json'
                )
        except urllib.error.HTTPError as exc:
            status = exc.code
            response_body = exc.read()
            error = repr(exc)
        except Exception as exc:
            error = repr(exc)
            response_body = json.dumps({'error': error}).encode()
        ended = time.perf_counter()
        usage = {}
        finish_reasons = []
        try:
            parsed = json.loads(response_body.decode('utf-8'))
            usage = parsed.get('usage') or {}
            finish_reasons = [
                choice.get('finish_reason')
                for choice in parsed.get('choices', [])
                if isinstance(choice, dict)
            ]
        except Exception:
            pass
        record = {
            'request_id': request_id,
            **context,
            'method': self.command,
            'path': self.path,
            'status_code': status,
            'started_perf': started,
            'ended_perf': ended,
            'request_seconds': ended - started,
            'usage': usage,
            'finish_reasons': finish_reasons,
            'error': error,
        }
        # Memory-only recording keeps disk writes outside inference timing.
        with proxy_lock:
            request_records.append(record)
        self.send_response(status)
        self.send_header('Content-Type', content_type)
        self.send_header('Content-Length', str(len(response_body)))
        self.end_headers()
        self.wfile.write(response_body)

proxy = ThreadingHTTPServer(('127.0.0.1', PROXY_PORT), ProxyHandler)
threading.Thread(target=proxy.serve_forever, daemon=True).start()
proxy_url = f'http://127.0.0.1:{PROXY_PORT}/v1'

def set_context(page_id=None, index=None, phase=None, attempt_id=None):
    with proxy_lock:
        proxy_state.update(
            page_id=page_id,
            manifest_index=index,
            phase=phase,
            attempt_id=attempt_id,
        )

def pop_request_records(attempt_ids):
    attempt_ids = set(attempt_ids)
    with proxy_lock:
        selected = [
            record for record in request_records
            if record.get('attempt_id') in attempt_ids
        ]
        request_records[:] = [
            record for record in request_records
            if record.get('attempt_id') not in attempt_ids
        ]
    return selected
''')

print('CLIENT WORKER PART 1: READY')


In [ ]:
# ================================================================
# CELL 5B — CLIENT WORKER: PAGE EXECUTION AND OUTPUTS
# ================================================================

WORKER_SOURCE_PARTS.append(r'''
def summarize_requests(records):
    posts = [record for record in records if record['method'] == 'POST']
    if not posts:
        return {
            'vlm_calls': 0,
            'vlm_seconds': 0.0,
            'vlm_request_seconds_sum': 0.0,
            'vlm_span_including_retry_gaps_seconds': 0.0,
            'generated_tokens': 0,
            'token_limit_reached': False,
            'vlm_http_error_count': 0,
            'vlm_timeout_count': 0,
        }
    grouped = {}
    for record in posts:
        grouped.setdefault(record['attempt_id'], []).append(record)
    attempt_spans = []
    for attempt_records in grouped.values():
        starts = [record['started_perf'] for record in attempt_records]
        ends = [record['ended_perf'] for record in attempt_records]
        attempt_spans.append(max(ends) - min(starts))
    generated_tokens = sum(
        int((record.get('usage') or {}).get('completion_tokens') or 0)
        for record in posts
    )
    errors = [str(record.get('error') or '') for record in posts]
    return {
        'vlm_calls': len(posts),
        'vlm_seconds': sum(attempt_spans),
        'vlm_request_seconds_sum': sum(
            record['request_seconds'] for record in posts
        ),
        'vlm_span_including_retry_gaps_seconds': (
            max(record['ended_perf'] for record in posts)
            - min(record['started_perf'] for record in posts)
        ),
        'generated_tokens': generated_tokens,
        'token_limit_reached': any(
            reason == 'length'
            for record in posts
            for reason in record.get('finish_reasons', [])
        ),
        'vlm_http_error_count': sum(
            int(record['status_code'] >= 400) for record in posts
        ),
        'vlm_timeout_count': sum(
            int('timed out' in error.lower() or 'timeout' in error.lower())
            for error in errors
        ),
    }

def save_result(entry, result):
    page_id = entry['page_id']
    raw_page_dir = RAW_PAGES_DIR / page_id
    raw_page_dir.mkdir(parents=True, exist_ok=True)
    raw_markdown = raw_page_dir / f'{page_id}.md'
    raw_json = raw_page_dir / f'{page_id}.json'
    result.save_to_markdown(
        str(raw_markdown), pretty=False, show_formula_number=False
    )
    result.save_to_json(str(raw_json))
    markdown_text = raw_markdown.read_text(encoding='utf-8')
    json.loads(raw_json.read_text(encoding='utf-8'))
    evaluator_path = EVALUATOR_DIR / f'{page_id}.md'
    # Keep the raw PaddleOCR Markdown untouched. In the evaluator copy,
    # change only asset URLs so they still resolve from evaluator_input.
    evaluator_text = markdown_text
    for old, new in {
        '(imgs/': f'(../raw_pages/{page_id}/imgs/',
        '(./imgs/': f'(../raw_pages/{page_id}/imgs/',
        'src="imgs/': f'src="../raw_pages/{page_id}/imgs/',
        'src="./imgs/': f'src="../raw_pages/{page_id}/imgs/',
    }.items():
        evaluator_text = evaluator_text.replace(old, new)
    atomic_text(evaluator_path, evaluator_text)
    return {
        'markdown_characters': len(evaluator_text),
        'markdown_nonempty': bool(evaluator_text.strip()),
        'markdown_sha256': hashlib.sha256(
            evaluator_text.encode('utf-8')
        ).hexdigest(),
        'raw_markdown_sha256': hashlib.sha256(
            markdown_text.encode('utf-8')
        ).hexdigest(),
        'evaluator_filename': evaluator_path.name,
        'raw_json_file': str(raw_json.relative_to(RESULT_DIR)),
    }

def save_empty_evaluator_output(entry):
    evaluator_path = EVALUATOR_DIR / f'{entry["page_id"]}.md'
    atomic_text(evaluator_path, '')
    return {
        'markdown_characters': 0,
        'markdown_nonempty': False,
        'markdown_sha256': hashlib.sha256(b'').hexdigest(),
        'evaluator_filename': evaluator_path.name,
        'raw_json_file': None,
    }

def run_page(index, phase, save_outputs):
    entry = INPUTS[index]
    page_id = entry['page_id']
    image_path = BUNDLE / entry['image']
    if image_path.stem != page_id or not image_path.is_file():
        raise RuntimeError(f'Input contract failure: {entry}')
    attempt_ids = []
    attempt_errors = []
    result = None
    elapsed = 0.0
    page_inference_started_utc = datetime.now(timezone.utc).isoformat()
    attempts_allowed = (
        int(RUNTIME['failure_policy']['retries_after_first_failure']) + 1
    )
    for attempt in range(attempts_allowed):
        attempt_id = f'{phase}-{page_id}-{attempt}-{time.time_ns()}'
        attempt_ids.append(attempt_id)
        set_context(page_id, index, phase, attempt_id)
        started = time.perf_counter()
        try:
            with paddle.no_grad():
                predictions = list(
                    pipeline.predict(str(image_path), **PREDICT_KWARGS)
                )
            paddle.device.synchronize()
            if len(predictions) != 1:
                raise RuntimeError(
                    f'Expected one prediction, got {len(predictions)}'
                )
            result = predictions[0]
            break
        except Exception:
            try:
                paddle.device.synchronize()
            except Exception:
                pass
            attempt_errors.append(traceback.format_exc())
        finally:
            elapsed += time.perf_counter() - started
            set_context()
    page_inference_ended_utc = datetime.now(timezone.utc).isoformat()

    records = pop_request_records(attempt_ids)
    request_summary = summarize_requests(records)
    # Disk persistence happens after elapsed inference timing is closed.
    append_jsonl(REQUEST_LOG_PATH, records)

    joined_errors = '\n\n'.join(attempt_errors) if attempt_errors else None
    timed_out = bool(
        joined_errors
        and ('timed out' in joined_errors.lower() or 'timeout' in joined_errors.lower())
    ) or request_summary['vlm_timeout_count'] > 0
    row = {
        'variant': VARIANT,
        'request_concurrency': CONCURRENCY,
        'phase': phase,
        'manifest_index': index,
        'page_id': page_id,
        'image_relative_path': entry['image'],
        'page_inference_seconds': elapsed,
        'page_inference_started_utc': page_inference_started_utc,
        'page_inference_ended_utc': page_inference_ended_utc,
        **request_summary,
        'attempts': len(attempt_ids),
        'retried': len(attempt_ids) > 1,
        'timed_out': timed_out,
        'status': 'completed' if result is not None else 'failed',
        'error': joined_errors,
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    }
    if result is not None:
        try:
            row['layout_blocks'] = len(result['parsing_res_list'])
        except Exception:
            row['layout_blocks'] = None
        if save_outputs:
            try:
                row.update(save_result(entry, result))
                row['output_status'] = 'saved'
                row['output_error'] = None
            except Exception:
                row.update(save_empty_evaluator_output(entry))
                row['output_status'] = 'failed_empty_placeholder_saved'
                row['output_error'] = traceback.format_exc()
                row['status'] = 'output_failed'
    elif save_outputs:
        row.update(save_empty_evaluator_output(entry))
        row['output_status'] = 'empty_placeholder_saved'
        row['output_error'] = None
    return row
''')

print('CLIENT WORKER PART 2: READY')


In [ ]:
# ================================================================
# CELL 5C — CLIENT WORKER: COMPLETE RUN AND SUMMARY
# ================================================================

WORKER_SOURCE_PARTS.append(r'''
def percentile(values, percentage):
    ordered = sorted(float(value) for value in values)
    if not ordered:
        return None
    position = (len(ordered) - 1) * percentage / 100.0
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1.0 - fraction) + ordered[upper] * fraction

try:
    startup_started = time.perf_counter()
    pipeline = PaddleOCRVL(
        pipeline_version='v1.6',
        device='gpu:0',
        layout_detection_model_name='PP-DocLayoutV3',
        layout_detection_model_dir=str(LAYOUT_DIR),
        vl_rec_backend='vllm-server',
        vl_rec_server_url=proxy_url,
        vl_rec_api_model_name=SERVED_NAME,
        vl_rec_max_concurrency=CONCURRENCY,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_layout_detection=True,
        use_chart_recognition=False,
        use_seal_recognition=False,
        use_ocr_for_image_block=False,
        use_queues=True,
        format_block_content=False,
        merge_layout_blocks=True,
        markdown_ignore_labels=[],
        enable_hpi=False,
        use_tensorrt=False,
        enable_cinn=False,
    )
    paddle.device.synchronize()
    client_startup_seconds = time.perf_counter() - startup_started

    atomic_json(RESULT_DIR / 'run_manifest.json', {
        'variant': VARIANT,
        'request_concurrency': CONCURRENCY,
        'client_startup_seconds': client_startup_seconds,
        'predict_kwargs': PREDICT_KWARGS,
        'warmup_indices': WARMUP_INDICES,
        'measured_indices': MEASURED_INDICES,
        'output_saving_excluded_from_page_timing': True,
        'request_log_disk_io_excluded_from_page_timing': True,
    })

    warmup_rows = []
    for index in WARMUP_INDICES:
        row = run_page(index, 'warmup', False)
        warmup_rows.append(row)
        print(
            'warmup', VARIANT, CONCURRENCY, index, row['status'],
            f"{row['page_inference_seconds']:.2f}s", flush=True,
        )
        if row['status'] != 'completed':
            raise RuntimeError(f'Warmup failed at index {index}')

    print('MEASUREMENT_READY', flush=True)
    gate_deadline = time.time() + 120
    while not MEASUREMENT_GATE.is_file():
        if time.time() >= gate_deadline:
            raise TimeoutError('Measurement monitor gate was not opened')
        time.sleep(0.05)

    measured_rows = []
    measurement_started_utc = datetime.now(timezone.utc).isoformat()
    measurement_started_perf = time.perf_counter()
    print('MEASUREMENT_START', flush=True)

    for position, index in enumerate(MEASURED_INDICES, start=1):
        row = run_page(index, 'measured', True)
        measured_rows.append(row)
        append_jsonl(METRICS_PATH, row)
        if (
            position == 1
            or position == len(MEASURED_INDICES)
            or position % PROGRESS_EVERY == 0
            or row['status'] != 'completed'
        ):
            print(
                'measured', VARIANT, CONCURRENCY,
                f'{position}/{len(MEASURED_INDICES)}', index,
                row['status'], f"{row['page_inference_seconds']:.2f}s",
                flush=True,
            )

    measurement_wall_seconds = time.perf_counter() - measurement_started_perf
    measurement_ended_utc = datetime.now(timezone.utc).isoformat()
    print('MEASUREMENT_END', flush=True)

    page_seconds = [row['page_inference_seconds'] for row in measured_rows]
    total_page_seconds = sum(page_seconds)
    total_vlm_seconds = sum(row['vlm_seconds'] for row in measured_rows)
    total_request_seconds = sum(
        row['vlm_request_seconds_sum'] for row in measured_rows
    )
    total_tokens = sum(row['generated_tokens'] for row in measured_rows)
    total_calls = sum(row['vlm_calls'] for row in measured_rows)
    completed = sum(row['status'] == 'completed' for row in measured_rows)
    failed = sum(row['status'] == 'failed' for row in measured_rows)
    output_failed = sum(row['status'] == 'output_failed' for row in measured_rows)
    retries = sum(bool(row['retried']) for row in measured_rows)
    timeouts = sum(bool(row['timed_out']) for row in measured_rows)
    token_limits = sum(
        bool(row['token_limit_reached']) for row in measured_rows
    )
    empty_outputs = sum(
        not bool(row.get('markdown_nonempty')) for row in measured_rows
    )

    summary_status = (
        'PASS' if failed == 0 and output_failed == 0
        else 'COMPLETED_WITH_FAILURES'
    )
    summary = {
        'variant': VARIANT,
        'request_concurrency': CONCURRENCY,
        'status': summary_status,
        'page_count': len(measured_rows),
        'completed_pages': completed,
        'failed_pages': failed,
        'output_failed_pages': output_failed,
        'retried_pages': retries,
        'timed_out_pages': timeouts,
        'token_limit_pages': token_limits,
        'empty_markdown_pages': empty_outputs,
        'success_rate': completed / len(measured_rows),
        'retry_rate': retries / len(measured_rows),
        'timeout_rate': timeouts / len(measured_rows),
        'token_limit_rate': token_limits / len(measured_rows),
        'empty_markdown_rate': empty_outputs / len(measured_rows),
        'client_startup_seconds': client_startup_seconds,
        'measurement_started_utc': measurement_started_utc,
        'measurement_ended_utc': measurement_ended_utc,
        'measurement_wall_seconds': measurement_wall_seconds,
        'total_page_inference_seconds': total_page_seconds,
        'mean_page_inference_seconds': statistics.fmean(page_seconds),
        'p50_page_inference_seconds': percentile(page_seconds, 50),
        'p90_page_inference_seconds': percentile(page_seconds, 90),
        'p95_page_inference_seconds': percentile(page_seconds, 95),
        'p99_page_inference_seconds': percentile(page_seconds, 99),
        'attempted_pages_per_hour': (
            len(measured_rows) * 3600.0 / total_page_seconds
        ),
        'successful_pages_per_hour': (
            completed * 3600.0 / total_page_seconds
        ),
        'observed_wall_pages_per_hour': (
            len(measured_rows) * 3600.0 / measurement_wall_seconds
        ),
        'total_vlm_seconds': total_vlm_seconds,
        'total_vlm_request_seconds_sum': total_request_seconds,
        'total_vlm_calls': total_calls,
        'total_generated_tokens': total_tokens,
        'tokens_per_vlm_second': (
            total_tokens / total_vlm_seconds if total_vlm_seconds else None
        ),
        'tokens_per_page_inference_second': (
            total_tokens / total_page_seconds if total_page_seconds else None
        ),
        'vlm_calls_per_page_inference_second': (
            total_calls / total_page_seconds if total_page_seconds else None
        ),
        'warmup_pages': warmup_rows,
        'measured_pages': measured_rows,
    }
    atomic_json(RESULT_DIR / 'measurement_window.json', {
        'started_utc': measurement_started_utc,
        'ended_utc': measurement_ended_utc,
        'wall_seconds': measurement_wall_seconds,
    })
    atomic_json(RESULT_DIR / 'speed_summary.json', summary)
    print(json.dumps({
        'variant': VARIANT,
        'request_concurrency': CONCURRENCY,
        'status': summary_status,
        'pages': len(measured_rows),
        'total_page_inference_seconds': total_page_seconds,
        'attempted_pages_per_hour': summary['attempted_pages_per_hour'],
        'tokens_per_vlm_second': summary['tokens_per_vlm_second'],
    }, indent=2), flush=True)
finally:
    proxy.shutdown()
    proxy.server_close()
''')

CLIENT_WORKER_SOURCE = '\n'.join(
    part.strip('\n') for part in WORKER_SOURCE_PARTS
) + '\n'
compile(CLIENT_WORKER_SOURCE, str(CLIENT_WORKER), 'exec')
CLIENT_WORKER.write_text(CLIENT_WORKER_SOURCE, encoding='utf-8')
if not CLIENT_WORKER.is_file() or CLIENT_WORKER.stat().st_size == 0:
    raise RuntimeError('Client worker was not created')

print('CLIENT WORKER: PASS', CLIENT_WORKER)


In [ ]:
# ================================================================
# CELL 6A — SERVER PROCESS AND LOG-AUDIT HELPERS
# ================================================================

def free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as stream:
        stream.bind(('127.0.0.1', 0))
        return stream.getsockname()[1]


def stop_process(process, interrupt_first=False):
    if process is None or process.poll() is not None:
        return
    if interrupt_first:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=60)
            return
        except subprocess.TimeoutExpired:
            pass
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=15)


def wait_for_server(port, process, log_path, timeout=1200):
    deadline = time.time() + timeout
    url = f'http://127.0.0.1:{port}/v1/models'
    while time.time() < deadline:
        if process.poll() is not None:
            tail = Path(log_path).read_text(
                encoding='utf-8', errors='ignore'
            )[-16000:]
            raise RuntimeError(
                f'vLLM exited during startup with {process.returncode}:\n{tail}'
            )
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    return
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f'vLLM startup timed out; see {log_path}')


def start_server(concurrency, log_path):
    port = free_local_port()
    served_name = VARIANT['id'].lower()
    command = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', str(CHECKPOINT),
        '--served-model-name', served_name,
        '--host', '127.0.0.1',
        '--port', str(port),
        '--trust-remote-code',
        '--dtype', 'bfloat16',
        '--tensor-parallel-size', '1',
        '--gpu-memory-utilization', str(
            RUNTIME_SOURCE['vllm']['gpu_memory_utilization']
        ),
        '--max-model-len', str(RUNTIME_SOURCE['vllm']['max_model_len']),
        '--max-num-seqs', '24',
        '--max-num-batched-tokens', str(
            RUNTIME_SOURCE['vllm']['max_num_batched_tokens']
        ),
        '--generation-config', 'vllm',
        '--no-enable-log-requests',
        '--no-enable-prefix-caching',
        '--mm-processor-cache-gb', '0',
        '--seed', '2026',
    ]
    if VARIANT['quantized']:
        command.extend(['--linear-backend', 'marlin'])

    environment = os.environ.copy()
    environment.update(offline_common)
    environment.update({
        'PYTHONPATH': str(SERVER_SITE),
        'CUDA_VISIBLE_DEVICES': '0',
        'VLLM_USE_FLASHINFER_SAMPLER': '0',
        'VLLM_LOGGING_LEVEL': 'INFO',
        'PYTHONUNBUFFERED': '1',
    })
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    stream = log_path.open('w', encoding='utf-8')
    print('+', ' '.join(map(str, command)), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        env=environment,
        stdout=stream,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
        wait_for_server(port, process, log_path)
    except Exception:
        stop_process(process)
        stream.close()
        raise
    return {
        'process': process,
        'stream': stream,
        'port': port,
        'served_name': served_name,
        'startup_seconds': time.perf_counter() - started,
        'command': command,
        'log_path': log_path,
        'concurrency': concurrency,
    }


def last_regex_groups(text, pattern):
    matches = list(re.finditer(pattern, text, re.IGNORECASE))
    return matches[-1].groupdict() if matches else None


def validate_server_log(log_path):
    text = Path(log_path).read_text(encoding='utf-8', errors='ignore')
    lower = text.lower()
    if 'traceback (most recent call last)' in lower:
        raise RuntimeError(f'Runtime traceback found in {log_path}')
    marlin_count = len(re.findall(
        r'Using MarlinLinearKernel for CompressedTensorsWNA16',
        text,
        re.IGNORECASE,
    ))
    if VARIANT['quantized']:
        if marlin_count < 1:
            raise RuntimeError('Native Marlin WNA16 selection not confirmed')
        if 'quantization=compressed-tensors' not in text:
            raise RuntimeError('Compressed-tensors runtime not confirmed')
        forbidden = ['MacheteLinearKernel', 'CutlassLinearKernel', 'BitBLASLinearKernel']
        hits = [token for token in forbidden if token.lower() in lower]
        if hits:
            raise RuntimeError(f'Unexpected quantized kernels: {hits}')
    elif marlin_count:
        raise RuntimeError('BF16 unexpectedly selected the Marlin WNA16 kernel')

    audit = {
        'requested_quantization': (
            f'GPTQ_{SELECTED_VARIANT}_G128' if VARIANT['quantized'] else None
        ),
        'resolved_quantization': (
            'compressed-tensors' if VARIANT['quantized'] else None
        ),
        'requested_linear_backend': (
            'marlin' if VARIANT['quantized'] else None
        ),
        'native_marlin_wna16_confirmed': (
            marlin_count >= 1 if VARIANT['quantized'] else None
        ),
        'marlin_selection_count': marlin_count,
        'logical_quantized_matrix_count': (
            234 if VARIANT['quantized'] else None
        ),
        'expected_fused_runtime_linear_modules': (
            126 if VARIANT['quantized'] else None
        ),
        'log_bytes': len(text.encode('utf-8')),
    }
    parsers = {
        'model_loading': (
            r'Model loading took (?P<memory_gib>[0-9.]+) GiB memory and '
            r'(?P<seconds>[0-9.]+) seconds'
        ),
        'available_kv_cache': (
            r'Available KV cache memory: (?P<memory_gib>[0-9.]+) GiB'
        ),
        'kv_cache_capacity': (
            r'GPU KV cache size: (?P<tokens>[0-9,]+) tokens, Maximum '
            r'concurrency for [0-9,]+ tokens per request: '
            r'(?P<maximum_concurrency>[0-9.]+)x'
        ),
        'engine_initialization': (
            r'init engine .*? took (?P<seconds>[0-9.]+) s '
            r'\(compilation: (?P<compilation_seconds>[0-9.]+) s\)'
        ),
        'profiled_memory': (
            r'Actual usage is (?P<consumed_gib>[0-9.]+) GiB for consumed '
            r'memory .*?, (?P<peak_activation_gib>[0-9.]+) GiB for peak '
            r'activation, and (?P<cudagraph_gib>[0-9.]+) GiB for CUDAGraph'
        ),
    }
    for key, pattern in parsers.items():
        audit[key] = last_regex_groups(text, pattern)
    if audit['model_loading'] is None or audit['engine_initialization'] is None:
        raise RuntimeError('Required vLLM memory/startup metrics missing from log')
    return audit

print('SERVER HELPERS: READY')


In [ ]:
# ================================================================
# CELL 6B — TELEMETRY, CLIENT PROCESS AND OUTPUT VALIDATION
# ================================================================

TELEMETRY_FIELDS = [
    'timestamp', 'index', 'uuid', 'utilization.gpu',
    'utilization.memory', 'power.draw', 'clocks.current.sm',
    'memory.used', 'memory.total', 'temperature.gpu', 'pstate',
]


def start_all_gpu_monitor(path):
    path = Path(path)
    stream = path.open('w', encoding='utf-8')
    command = [
        'nvidia-smi',
        '--query-gpu=' + ','.join(TELEMETRY_FIELDS),
        '--format=csv,noheader,nounits',
        '-lms', str(TELEMETRY_INTERVAL_MS),
    ]
    process = subprocess.Popen(
        command,
        stdout=stream,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return process, stream


def percentile(values, percentage):
    ordered = sorted(float(value) for value in values)
    if not ordered:
        return None
    position = (len(ordered) - 1) * percentage / 100.0
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1.0 - fraction) + ordered[upper] * fraction


def parse_float(value):
    try:
        return float(value.strip())
    except (TypeError, ValueError):
        return None


def parse_nvidia_timestamp(value):
    value = value.strip()
    for pattern in ['%Y/%m/%d %H:%M:%S.%f', '%Y/%m/%d %H:%M:%S']:
        try:
            return datetime.strptime(value, pattern).replace(
                tzinfo=timezone.utc
            )
        except ValueError:
            pass
    return None


def summarize_telemetry(path, measured_pages, total_page_seconds):
    page_count = len(measured_pages)
    intervals = sorted(
        (
            datetime.fromisoformat(row['page_inference_started_utc']),
            datetime.fromisoformat(row['page_inference_ended_utc']),
        )
        for row in measured_pages
    )
    interval_starts = [start for start, _ in intervals]

    def inside_inference_interval(sample_time):
        position = bisect.bisect_right(interval_starts, sample_time) - 1
        return (
            position >= 0
            and sample_time <= intervals[position][1]
        )

    grouped = {index: [] for index in range(4)}
    total_samples = {index: 0 for index in range(4)}
    for fields in csv.reader(
        Path(path).read_text(encoding='utf-8', errors='ignore').splitlines()
    ):
        if len(fields) != len(TELEMETRY_FIELDS):
            continue
        try:
            index = int(fields[1].strip())
        except ValueError:
            continue
        if index not in grouped:
            continue
        total_samples[index] += 1
        sample_time = parse_nvidia_timestamp(fields[0])
        if (
            sample_time is None
            or not inside_inference_interval(sample_time)
        ):
            continue
        grouped[index].append({
            'gpu_utilization_percent': parse_float(fields[3]),
            'memory_utilization_percent': parse_float(fields[4]),
            'power_watts': parse_float(fields[5]),
            'sm_clock_mhz': parse_float(fields[6]),
            'memory_used_mib': parse_float(fields[7]),
            'memory_total_mib': parse_float(fields[8]),
            'temperature_c': parse_float(fields[9]),
            'pstate': fields[10].strip(),
        })

    roles = {0: 'vllm_server', 1: 'paddle_layout', 2: 'unused', 3: 'unused'}
    summary = {}
    for index, rows in grouped.items():
        if len(rows) < 2:
            raise RuntimeError(
                f'Insufficient telemetry samples for GPU {index}: {len(rows)}'
            )

        def values(key):
            return [row[key] for row in rows if row[key] is not None]

        gpu_util = values('gpu_utilization_percent')
        memory_used = values('memory_used_mib')
        power = values('power_watts')
        clocks = values('sm_clock_mhz')
        temperatures = values('temperature_c')
        energy_joules = (
            statistics.fmean(power) * total_page_seconds
            if power else None
        )
        summary[f'gpu_{index}'] = {
        'role': roles[index],
        'samples': len(rows),
        'raw_samples_before_window_filter': total_samples[index],
        'sampling_interval_ms': TELEMETRY_INTERVAL_MS,
        'window_filter': 'union of per-page primary inference intervals',
            'gpu_utilization_mean_percent': statistics.fmean(gpu_util),
            'gpu_utilization_p50_percent': percentile(gpu_util, 50),
            'gpu_utilization_p95_percent': percentile(gpu_util, 95),
            'gpu_utilization_max_percent': max(gpu_util),
            'gpu_nonzero_fraction': (
                sum(value > 0 for value in gpu_util) / len(gpu_util)
            ),
            'memory_used_min_mib': min(memory_used),
            'memory_used_max_mib': max(memory_used),
            'memory_used_peak_delta_mib': max(memory_used) - min(memory_used),
            'power_mean_watts': statistics.fmean(power) if power else None,
            'power_p95_watts': percentile(power, 95) if power else None,
            'power_max_watts': max(power) if power else None,
        'inference_energy_joules_estimate': energy_joules,
        'inference_energy_wh_estimate': (
                energy_joules / 3600.0 if energy_joules is not None else None
            ),
            'energy_joules_per_attempted_page': (
                energy_joules / page_count
                if energy_joules is not None and page_count else None
            ),
            'sm_clock_mean_mhz': statistics.fmean(clocks) if clocks else None,
            'sm_clock_p50_mhz': percentile(clocks, 50) if clocks else None,
            'sm_clock_p95_mhz': percentile(clocks, 95) if clocks else None,
            'temperature_mean_c': (
                statistics.fmean(temperatures) if temperatures else None
            ),
            'temperature_max_c': max(temperatures) if temperatures else None,
            'pstates_observed': sorted({row['pstate'] for row in rows}),
        }
    return summary


def validate_evaluator_outputs(result_dir, summary):
    result_dir = Path(result_dir)
    evaluator_dir = result_dir / 'evaluator_input'
    expected_names = {
        benchmark_inputs[index]['page_id'] + '.md'
        for index in MEASURED_INDICES
    }
    flat_files = sorted(evaluator_dir.glob('*.md'))
    recursive_files = sorted(evaluator_dir.rglob('*.md'))
    actual_names = {path.name for path in flat_files}
    if actual_names != expected_names:
        raise RuntimeError(
            'Evaluator filename drift: '
            f'missing={sorted(expected_names - actual_names)[:20]}, '
            f'extra={sorted(actual_names - expected_names)[:20]}'
        )
    if flat_files != recursive_files:
        raise RuntimeError('Nested Markdown found in evaluator_input')

    rows = summary.get('measured_pages') or []
    indices = [int(row['manifest_index']) for row in rows]
    if indices != MEASURED_INDICES:
        raise RuntimeError('Measured-page order or coverage drift')
    if len(indices) != len(set(indices)):
        raise RuntimeError('Duplicate measured-page record')

    file_manifest = []
    for path in flat_files:
        data = path.read_bytes()
        file_manifest.append({
            'filename': path.name,
            'bytes': len(data),
            'sha256': hashlib.sha256(data).hexdigest(),
            'nonempty': bool(data.strip()),
        })
    empty_count = sum(not row['nonempty'] for row in file_manifest)
    contract = {
        'status': (
            'PASS' if empty_count == 0
            else 'INVALID_FOR_QUALITY_EVALUATION_EMPTY_OUTPUTS'
        ),
        'directory': str(evaluator_dir),
        'expected_files': len(expected_names),
        'actual_files': len(flat_files),
        'unique_filenames': len(actual_names),
        'empty_files': empty_count,
        'flat_directory_only': True,
        'filenames_match_omnidocbench_image_stems': True,
        'files': file_manifest,
    }
    (result_dir / 'evaluator_output_manifest.json').write_text(
        json.dumps(contract, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    if RUN_MODE == 'PREFLIGHT' and empty_count:
        raise RuntimeError(
            f'Preflight produced {empty_count} empty evaluator outputs'
        )
    return contract


def run_client(server, concurrency, result_dir):
    result_dir = Path(result_dir)
    result_dir.mkdir(parents=True, exist_ok=False)
    client_log_path = result_dir / 'client.log'
    telemetry_path = result_dir / 'all_gpu_telemetry.csv'
    gate_path = result_dir / 'measurement_monitor_ready'
    gate_path.unlink(missing_ok=True)

    environment = os.environ.copy()
    environment.update(offline_common)
    environment.update({
        'PYTHONPATH': str(CLIENT_SITE),
        'CUDA_VISIBLE_DEVICES': '1',
        'BUNDLE': str(BUNDLE),
        'RESULT_DIR': str(result_dir),
        'LAYOUT_DIR': str(LAYOUT),
        'SERVER_URL': f'http://127.0.0.1:{server["port"]}/v1',
        'PROXY_PORT': str(free_local_port()),
        'SERVED_NAME': server['served_name'],
        'VARIANT': VARIANT['id'],
        'REQUEST_CONCURRENCY': str(concurrency),
        'RUNTIME_CONTRACT': str(BUNDLE / MANIFEST['paths']['runtime_contract']),
        'BENCHMARK_INPUTS': str(BENCHMARK_INPUTS),
        'WARMUP_INDICES_JSON': json.dumps(WARMUP_INDICES),
        'MEASURED_INDICES_JSON': json.dumps(MEASURED_INDICES),
        'EXPECTED_DATASET_PAGES': str(EXPECTED_PAGES),
        'MEASUREMENT_GATE': str(gate_path),
        'PROGRESS_EVERY': '25' if RUN_MODE == 'FULL' else '1',
    })

    command = [sys.executable, str(CLIENT_WORKER)]
    monitor = monitor_stream = None
    ready_seen = start_seen = end_seen = False
    return_code = None

    with client_log_path.open('w', encoding='utf-8') as client_log:
        client = subprocess.Popen(
            command,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if client.stdout is None:
            stop_process(client)
            raise RuntimeError('Client stdout pipe is unavailable')
        try:
            for line in client.stdout:
                print(line, end='', flush=True)
                client_log.write(line)
                client_log.flush()
                marker = line.strip()
                if marker == 'MEASUREMENT_READY':
                    if ready_seen:
                        raise RuntimeError('Duplicate MEASUREMENT_READY marker')
                    ready_seen = True
                    monitor, monitor_stream = start_all_gpu_monitor(telemetry_path)
                    time.sleep(max(0.5, TELEMETRY_INTERVAL_MS / 1000.0 * 2))
                    if monitor.poll() is not None:
                        raise RuntimeError('nvidia-smi monitor exited before measurement')
                    gate_path.write_text('ready\n', encoding='utf-8')
                elif marker == 'MEASUREMENT_START':
                    start_seen = True
                elif marker == 'MEASUREMENT_END':
                    end_seen = True
                    stop_process(monitor)
                    if monitor_stream is not None:
                        monitor_stream.close()
                    monitor = monitor_stream = None
            return_code = client.wait()
        except Exception:
            stop_process(client)
            raise
        finally:
            if monitor is not None:
                stop_process(monitor)
            if monitor_stream is not None:
                monitor_stream.close()

    if return_code != 0:
        tail = client_log_path.read_text(
            encoding='utf-8', errors='ignore'
        )[-16000:]
        raise RuntimeError(
            f'Client failed at c{concurrency} with {return_code}:\n{tail}'
        )
    if not (ready_seen and start_seen and end_seen):
        raise RuntimeError(
            f'Measurement marker failure: ready={ready_seen}, '
            f'start={start_seen}, end={end_seen}'
        )

    summary_path = result_dir / 'speed_summary.json'
    summary = load_json(summary_path)
    telemetry = summarize_telemetry(
        telemetry_path,
        summary['measured_pages'],
        summary['total_page_inference_seconds'],
    )
    output_contract = validate_evaluator_outputs(result_dir, summary)
    summary['gpu_telemetry'] = telemetry
    summary['output_contract'] = {
        key: value for key, value in output_contract.items() if key != 'files'
    }
    summary_path.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    return summary

print('CLIENT + TELEMETRY HELPERS: READY')


In [ ]:
# ================================================================
# CELL 7 — RUN c1 AND c24 WITH INDEPENDENT vLLM SERVERS
# ================================================================

def gpu_snapshot():
    return subprocess.check_output(
        [
            'nvidia-smi',
            '--query-gpu=index,name,uuid,utilization.gpu,memory.used,'
            'memory.total,power.draw,temperature.gpu,pstate',
            '--format=csv,noheader,nounits',
        ],
        text=True,
    )


completed_runs = []

for concurrency in CONCURRENCIES:
    run_dir = OUTPUT_ROOT / f'c{concurrency}'
    server_log = OUTPUT_ROOT / f'vllm_c{concurrency}.log'
    (OUTPUT_ROOT / f'gpu_before_c{concurrency}.txt').write_text(
        gpu_snapshot(), encoding='utf-8'
    )

    print('=' * 100)
    print(
        f'STARTING {VARIANT["id"]} c{concurrency} '
        f'({RUN_MODE}, {len(MEASURED_INDICES)} measured pages)'
    )
    print('=' * 100)

    server = None
    try:
        server = start_server(concurrency, server_log)
        startup_audit = validate_server_log(server_log)
        summary = run_client(server, concurrency, run_dir)

        # Validate again while the server is alive. Shutdown-only vLLM
        # messages cannot be mistaken for measurement failures.
        final_server_audit = validate_server_log(server_log)
        summary['server_startup_seconds'] = server['startup_seconds']
        summary['server_command'] = server['command']
        summary['server_startup_audit'] = startup_audit
        summary['server_final_audit'] = final_server_audit
        summary['gpu_assignment'] = {
            'vllm_server_physical_gpu': 0,
            'paddle_layout_physical_gpu': 1,
            'unused_physical_gpus': [2, 3],
        }
        (run_dir / 'speed_summary.json').write_text(
            json.dumps(summary, indent=2, ensure_ascii=False) + '\n',
            encoding='utf-8',
        )
        completed_runs.append(summary)
    finally:
        if server is not None:
            stop_process(server['process'], interrupt_first=True)
            server['stream'].close()

    time.sleep(3)
    (OUTPUT_ROOT / f'gpu_after_c{concurrency}.txt').write_text(
        gpu_snapshot(), encoding='utf-8'
    )
    print(f'COMPLETED c{concurrency}', flush=True)

if len(completed_runs) != 2:
    raise RuntimeError(f'Expected two completed runs, got {len(completed_runs)}')
if [run['request_concurrency'] for run in completed_runs] != [1, 24]:
    raise RuntimeError('Completed concurrency order drift')

print('INDEPENDENT c1 + c24 RUNS: PASS')


In [ ]:
# ================================================================
# CELL 8 — FINAL VALIDATION, REPORT, ARCHIVE AND SAFE CLEANUP
# ================================================================

run_rows = []
for run in completed_runs:
    if run['page_count'] != len(MEASURED_INDICES):
        raise RuntimeError('Run page count drift')
    if run['output_contract']['actual_files'] != len(MEASURED_INDICES):
        raise RuntimeError('Evaluator output count drift')
    if run['status'] not in {'PASS', 'COMPLETED_WITH_FAILURES'}:
        raise RuntimeError(f'Unexpected run status: {run["status"]}')
    if RUN_MODE == 'PREFLIGHT' and run['status'] != 'PASS':
        raise RuntimeError(f'Preflight c{run["request_concurrency"]} failed')

    server_gpu = run['gpu_telemetry']['gpu_0']
    layout_gpu = run['gpu_telemetry']['gpu_1']
    load_metric = run['server_final_audit']['model_loading'] or {}
    run_rows.append({
        'variant': run['variant'],
        'run_mode': RUN_MODE,
        'request_concurrency': run['request_concurrency'],
        'page_count': run['page_count'],
        'status': run['status'],
        'mean_page_inference_seconds': run['mean_page_inference_seconds'],
        'p50_page_inference_seconds': run['p50_page_inference_seconds'],
        'p90_page_inference_seconds': run['p90_page_inference_seconds'],
        'p95_page_inference_seconds': run['p95_page_inference_seconds'],
        'p99_page_inference_seconds': run['p99_page_inference_seconds'],
        'attempted_pages_per_hour': run['attempted_pages_per_hour'],
        'successful_pages_per_hour': run['successful_pages_per_hour'],
        'tokens_per_vlm_second': run['tokens_per_vlm_second'],
        'tokens_per_page_inference_second': (
            run['tokens_per_page_inference_second']
        ),
        'success_rate': run['success_rate'],
        'retry_rate': run['retry_rate'],
        'timeout_rate': run['timeout_rate'],
        'token_limit_rate': run['token_limit_rate'],
        'empty_markdown_rate': run['empty_markdown_rate'],
        'server_startup_seconds': run['server_startup_seconds'],
        'client_startup_seconds': run['client_startup_seconds'],
        'vllm_model_loading_memory_gib': load_metric.get('memory_gib'),
        'vllm_model_loading_seconds': load_metric.get('seconds'),
        'server_gpu_utilization_mean_percent': (
            server_gpu['gpu_utilization_mean_percent']
        ),
        'server_memory_used_max_mib': server_gpu['memory_used_max_mib'],
        'server_power_mean_watts': server_gpu['power_mean_watts'],
        'server_energy_joules_per_attempted_page': (
            server_gpu['energy_joules_per_attempted_page']
        ),
        'layout_gpu_utilization_mean_percent': (
            layout_gpu['gpu_utilization_mean_percent']
        ),
        'layout_memory_used_max_mib': layout_gpu['memory_used_max_mib'],
    })

omnidocbench_evaluation_ready = (
    RUN_MODE == 'FULL'
    and len(MEASURED_INDICES) == EXPECTED_PAGES
    and all(
        run['status'] == 'PASS'
        and run['output_contract']['status'] == 'PASS'
        and run['output_contract']['actual_files'] == EXPECTED_PAGES
        for run in completed_runs
    )
)

report = {
    'study_id': STUDY['study_id'],
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'PASS' if all(run['status'] == 'PASS' for run in completed_runs)
    else 'COMPLETED_WITH_FAILURES',
    'run_mode': RUN_MODE,
    'variant': selected_contract,
    'topology': {
        'gpu_0': 'vllm_server',
        'gpu_1': 'PP-DocLayoutV3',
        'gpu_2': 'intentionally_unused',
        'gpu_3': 'intentionally_unused',
    },
    'controls': {
        'request_concurrencies': [1, 24],
        'fresh_vllm_server_per_concurrency': True,
        'prefix_cache_disabled': True,
        'multimodal_processor_cache_disabled': True,
        'output_saving_excluded_from_primary_page_timing': True,
        'request_log_disk_io_excluded_from_primary_page_timing': True,
        'generation': RUNTIME_SOURCE['generation'],
    },
    'hardware': [
        {
            'index': row[0].strip(),
            'name': row[1].strip(),
            'compute_capability': row[2].strip(),
            'memory_mib': row[3].strip(),
            'uuid': row[4].strip(),
            'driver_version': row[5].strip(),
        }
        for row in gpu_rows
    ],
    'host': {
        'python': sys.version,
        'platform': platform.platform(),
        'processor': platform.processor(),
    },
    'software_versions': SOFTWARE_VERSIONS,
    'omnidocbench_evaluation_ready': omnidocbench_evaluation_ready,
    'runs': run_rows,
}
(OUTPUT_ROOT / 'benchmark_summary.json').write_text(
    json.dumps(report, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
(OUTPUT_ROOT / 'study_design.json').write_text(
    json.dumps(STUDY, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
(OUTPUT_ROOT / 'source_runtime_contract.json').write_text(
    json.dumps(RUNTIME_SOURCE, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
evaluation_contract = {
    'recommended_method': 'end2end',
    'recommended_match_method': 'quick_match',
    'ground_truth_json_in_attached_bundle': str(GROUND_TRUTH),
    'prediction_directories': {
        'concurrency_1': 'c1/evaluator_input',
        'concurrency_24': 'c24/evaluator_input',
    },
    'filename_rule': (
        'OmniDocBench image basename with its image extension '
        'replaced by .md'
    ),
    'full_dataset_pages': EXPECTED_PAGES,
    'evaluation_ready': omnidocbench_evaluation_ready,
    'do_not_report_quality_if_false': True,
}
(OUTPUT_ROOT / 'omnidocbench_evaluation_contract.json').write_text(
    json.dumps(evaluation_contract, indent=2) + '\n',
    encoding='utf-8',
)

csv_path = OUTPUT_ROOT / 'run_metrics.csv'
with csv_path.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(run_rows[0]))
    writer.writeheader()
    writer.writerows(run_rows)

readme = '\n'.join([
    'PaddleOCR-VL-1.6 OmniDocBench predictions',
    '',
    f"Variant: {VARIANT['id']}",
    f'Mode: {RUN_MODE}',
    f'Pages per concurrency: {len(MEASURED_INDICES)}',
    'Concurrencies: 1 and 24',
    f'OmniDocBench evaluation ready: {omnidocbench_evaluation_ready}',
    '',
    'For OmniDocBench Docker evaluation, use these flat prediction directories:',
    '  c1/evaluator_input',
    '  c24/evaluator_input',
    '',
    'Each filename is the exact OmniDocBench image stem with the .md extension.',
    'The evaluator_output_manifest.json file in each run contains SHA-256 hashes.',
    'Do not report a quality score unless evaluation_ready is true.',
    '',
])
(OUTPUT_ROOT / 'README.txt').write_text(readme, encoding='utf-8')

archive_base = Path(str(OUTPUT_ROOT) + '_results')
archive_path = Path(shutil.make_archive(
    str(archive_base), 'zip', root_dir=OUTPUT_ROOT
))
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise RuntimeError('Result archive was not created')
with zipfile.ZipFile(archive_path) as archive:
    corrupt = archive.testzip()
    members = archive.namelist()
if corrupt is not None:
    raise RuntimeError(f'Corrupt archive member: {corrupt}')
required_archive_members = {
    'benchmark_summary.json',
    'c1/speed_summary.json',
    'c24/speed_summary.json',
}
if not required_archive_members.issubset(set(members)):
    raise RuntimeError('Required files are missing from the result archive')

# Only disposable installed environments are removed. Results, raw page
# outputs, evaluator inputs, logs, telemetry and archives are preserved.
if CLEAN_TEMP_ENVIRONMENTS_AFTER_RUN:
    for disposable in [SERVER_SITE, CLIENT_SITE]:
        shutil.rmtree(disposable, ignore_errors=True)

print('=' * 100)
if RUN_MODE == 'PREFLIGHT':
    print('PREFLIGHT: PASS')
    print("Next: change RUN_MODE = 'FULL' and run from the beginning.")
elif report['status'] == 'PASS':
    print('FULL BENCHMARK: PASS')
else:
    print('FULL BENCHMARK: COMPLETED WITH FAILURES — REVIEW REPORT')
print('Variant:', VARIANT['id'])
print('Result directory:', OUTPUT_ROOT)
print('Result archive:', archive_path)
print('Archive size MiB:', round(archive_path.stat().st_size / 2**20, 2))
print('=' * 100)

try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive_path)))
except Exception:
    pass
